# 03 - Enrich Crop + Weather (Silver)

Combines the two clean tables into `silver.crop_production_weather`.

1. Convert the crop file's UPPERCASE province name to the weather file's style (`AGUSAN DEL NORTE` -> `Agusan del Norte`)
2. Left-join the weather data on standard province + year + quarter

**Column naming in the enriched table**
- `province` = standard (Title Case) name, used by every Gold table
- `province_source` = original UPPERCASE name from the crop file
- `weather_*` = columns from the weather file. The crop file has its own climate columns with different
  values, so both sets are kept.
- `has_weather_match` = `true` when a weather row was found

## 1. Configuration

In [0]:
from pyspark.sql import functions as F

from common.delta_io import write_delta_table
from configs.etl_config import SILVER_CROP_TABLE, SILVER_CROP_WEATHER_TABLE, SILVER_WEATHER_TABLE

## 2. Standard province name

Capitalize every word, then put the small words back in lowercase (`del`, `de`, ...) unless they start the name

In [0]:
lowercase_words = ["del", "de", "la", "las", "los", "y", "van", "von"]

standard_name = F.initcap(F.lower(F.col("province_source")))
for word in lowercase_words:
    standard_name = F.regexp_replace(standard_name, f"(?<=\\s){word.capitalize()}(?=\\s|$)", word)

crop_df = (
    spark.table(SILVER_CROP_TABLE)
    .withColumnRenamed("province", "province_source")
    .withColumn("province", standard_name)
)

# Quick visual check of the conversion
crop_df.select("province_source", "province").distinct().orderBy("province_source").show(60, truncate=False)

## 3. Join the weather data

In [0]:
weather_df = spark.table(SILVER_WEATHER_TABLE)

# Prefix every weather column with "weather_" so nothing clashes with the crop file's own climate columns
weather_prefixed_df = weather_df.select(
    *[F.col(column).alias(f"weather_{column}") for column in weather_df.columns]
)

enriched_df = (
    crop_df
    .join(
        weather_prefixed_df,
        (F.col("province") == F.col("weather_province"))
        & (F.col("year") == F.col("weather_year"))
        & (F.col("quarter") == F.col("weather_quarter")),
        "left",
    )
    .withColumn("has_weather_match", F.col("weather_province").isNotNull())
    .drop("weather_province", "weather_year", "weather_quarter")
)

## 4. Write the enriched table

In [0]:
write_delta_table(enriched_df, SILVER_CROP_WEATHER_TABLE)

enriched_check_df = spark.table(SILVER_CROP_WEATHER_TABLE)
total_rows = enriched_check_df.count()
matched_rows = enriched_check_df.filter(F.col("has_weather_match")).count()
print(f"Rows: {total_rows:,} | with weather match: {matched_rows:,} ({100 * matched_rows / total_rows:.1f}%)")
print("Silver complete. Next: 03_gold/01_province_quarter_summary")